In [1]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy import constants, units
from astropy.coordinates import Angle
from scipy.optimize import curve_fit
import pandas as pd
import os
from scipy.signal import medfilt
from astropy.wcs import WCS
import scipy.integrate as integrate

In [2]:
rootdir = '/Users/thepoetoftwilight/Documents/CUBS/Data/PG1522+101/HST_images/'

Define utility functions to compute photometry for an HST image based on its FITS file and the segmentation map/ catalog

In [3]:
def hst_photometry_compute(img_fits, seg_fits, cat_file):
    
    # Create a WCS coordinate system for the image header
    wcs = WCS(header=img_fits[0].header)
    
    # Get the segmentation map
    seg_map = seg_fits[0].data
    
    # Get coordinates for each identified galaxy in the segmentation map
    gal_ids_arr = np.int_(cat_file[:,0])

    gal_ids_coords = {}
    
    for i in range(len(gal_ids_arr)):

        # Isolate the galaxy ID
        gal_id = gal_ids_arr[i]

        # Get x, y coordinates of the galaxy
        # Once again, remember the transposition
        seg_map_gal_y, seg_map_gal_x = np.where(seg_map==gal_id)

        gal_ids_coords[gal_id] = []

        for j in range(len(seg_map_gal_x)):

            # Append the galactic coordinates
            gal_ids_coords[gal_id].append([seg_map_gal_y[j], seg_map_gal_x[j]])

        gal_ids_coords[gal_id] = np.array(gal_ids_coords[gal_id])
        
    # Add up fluxes for all galaxies and compute apparent magnitude
    
    flux_map = img_fits[0].data*img_fits[0].header['PHOTFNU']
    
    gal_ids_flux = np.zeros(len(gal_ids_arr))
    
    for i in range(len(gal_ids_arr)):

        # Get coordinates for the current galaxy
        gal_coords = gal_ids_coords[gal_ids_arr[i]]

        for j in range(gal_coords.shape[0]):

            # Add up fluxes for each coordinate
            coords = gal_coords[j]
            gal_ids_flux[i] += flux_map[coords[0], coords[1]]
            
    gal_ids_mag = -2.5*np.log10(gal_ids_flux/3631)
    
    return gal_ids_mag

Load in the image, segmentation map, and catalog file for F160W

In [4]:
img_fits_160w = fits.open(rootdir+'f160w_comb.fits')
seg_fits_160w = fits.open(rootdir+'f160w_seg.fits')
cat_file_160w = np.loadtxt(rootdir + 'f160w.cat')

In [5]:
img_fits_140w = fits.open(rootdir+'f140w_comb.fits')
seg_fits_140w = fits.open(rootdir+'f140w_seg.fits')
cat_file_140w = np.loadtxt(rootdir + 'f140w.cat')

In [6]:
gal_ids_mag_160w = hst_photometry_compute(img_fits_160w, seg_fits_160w, cat_file_160w)

the RADECSYS keyword is deprecated, use RADESYSa. [astropy.wcs.wcs]
/var/folders/tj/vc_wjrpj36sf3zws4s7s770c0000gn/T/ipykernel_7847/905744176.py:49: RuntimeWarning: invalid value encountered in log10
  gal_ids_mag = -2.5*np.log10(gal_ids_flux/3631)


In [7]:
gal_ids_mag_140w = hst_photometry_compute(img_fits_140w, seg_fits_140w, cat_file_140w)

/var/folders/tj/vc_wjrpj36sf3zws4s7s770c0000gn/T/ipykernel_7847/905744176.py:49: RuntimeWarning: invalid value encountered in log10
  gal_ids_mag = -2.5*np.log10(gal_ids_flux/3631)


Write up a catalog for each image

In [8]:
def write_catalog(cat_file, gal_ids_mag, fname, mag_col_name):
    
    # Galaxy IDs (can help extract the spectra)
    gal_ids_arr = cat_file[:,0]
    
    # Galaxy coordinates, first load in degrees
    gal_ra_arr_deg = cat_file[:,1]
    gal_dec_arr_deg = cat_file[:,2]
    
    gal_x_arr = cat_file[:,3]
    gal_y_arr = cat_file[:,4]

    gal_ra_arr_hms = [Angle(ra, units.degree).hms for ra in gal_ra_arr_deg]
    gal_dec_arr_dms = [Angle(dec, units.degree).dms for dec in gal_dec_arr_deg]

    gal_ra_arr_hms_str = np.array([str(int(ra[0])) + 'h' + str(int(ra[1])) + 'm' + str(np.round(ra[2],2)) + 's' 
                     for ra in gal_ra_arr_hms])

    gal_dec_arr_dms_str = np.array([str(int(dec[0])) + 'd' + str(int(dec[1])) + 'm' + str(np.round(dec[2],2)) + 's' 
                     for dec in gal_dec_arr_dms])
    
    # Save the file name
    
    with open(fname, 'w') as f:
        
        f.write('ID,RA,Dec,x,y,'+mag_col_name)      

        # Write out the catalog
        for i in range(len(gal_ids_arr)):

            f.write('\n'+str(int(gal_ids_arr[i])) + ',' + 
                  str(gal_ra_arr_deg[i]) + ',' + 
                  str(gal_dec_arr_deg[i]) + ',' + 
                  str(gal_x_arr[i]) + ',' + 
                  str(gal_y_arr[i]) + ',' + 
                  str('%.2f'%np.round(gal_ids_mag[i],2)))

In [9]:
write_catalog(cat_file_160w, gal_ids_mag_160w, rootdir+'f160w_photometry.dat', 'f160w_mag')

In [10]:
write_catalog(cat_file_140w, gal_ids_mag_140w, rootdir+'f140w_photometry.dat', 'f140w_mag')